# 04 · Replication sample (QPN)

Same analysis as `02_software_variability` on the Quebec Parkinson Network (QPN) sample, processed outside
these notebooks with the same pipeline (FS5, FS6, FS7; **FS8 not processed yet**).

**Not executable yet**: the QPN inputs (`data/qpn/QPN_subjects.csv`, `data/qpn/stats/FSx/<subject>/stats/`)
are not in `/lustre09/project/6049200/vinuyans` — locate them and link them as `data/qpn` (see `TODO.md`).

In [ ]:
import json

import pandas as pd
from scipy.stats import ttest_ind

import fsvv

QPN_DIR = fsvv.DATA / "qpn"
if not (QPN_DIR / "QPN_subjects.csv").exists():
    raise FileNotFoundError(f"QPN inputs missing: link them as {QPN_DIR} (see TODO.md)")

QPN_VERSIONS = [v for v in fsvv.VERSIONS if (QPN_DIR / "stats" / v).is_dir()]
QPN_PAIRS = [p for p in fsvv.PAIRS if set(p) <= set(QPN_VERSIONS)]
print("QPN versions:", QPN_VERSIONS)

qpn = pd.read_csv(QPN_DIR / "QPN_subjects.csv")[["PATNO", "sex", "age"]]
print(f"{(qpn['sex'] == 'F').sum()} women, {(qpn['sex'] == 'M').sum()} men; "
      f"age {qpn['age'].mean():.1f} ± {qpn['age'].std():.1f}")

## Measures, version differences and paired t-tests

In [ ]:
measures = fsvv.extract_measures(qpn["PATNO"], QPN_VERSIONS, root=QPN_DIR / "stats")
qpn_table = pd.concat([measures, fsvv.version_differences(measures, QPN_PAIRS)], axis=1)
tests = fsvv.paired_version_tests(qpn_table, QPN_PAIRS)
tests.to_csv(fsvv.out("qpn_version_paired_ttests.csv"), index=False)
sets = fsvv.significant_sets(tests)
display(fsvv.proportion_significant(sets, QPN_PAIRS).mul(100).round(1))  # %
for kind in fsvv.CORTICAL_MEASURES:
    display(fsvv.dice_between_sets(sets, kind, QPN_PAIRS).round(2))

## Software variability: QPN vs PPMI HC

In [ ]:
ppmi = pd.read_csv(fsvv.inp("aseg_table4.csv")).set_index("PATNO_id")
ppmi_hc = ppmi[ppmi["dx_group"] == "HC"]
both = pd.concat([qpn_table.assign(sample="QPN"), ppmi_hc[qpn_table.columns.intersection(ppmi_hc.columns)].assign(sample="PPMI-HC")])
qpn_vs_hc = fsvv.group_variability_tests(both, both["sample"] == "QPN", both["sample"] == "PPMI-HC", QPN_PAIRS)
qpn_vs_hc["significant"] = qpn_vs_hc["p"] < fsvv.ALPHA / qpn_vs_hc["kind"].map(fsvv.N_TESTS)
qpn_vs_hc.groupby(["pair", "kind"])["significant"].sum().unstack("kind")

## MRIQC (QPN) and correlation with software variability

In [ ]:
mriqc_dir = fsvv.DATA / "source_files" / "clinical" / "mriqc" / "output_qpn"
iqm = {s: json.load(open(mriqc_dir / f"sub-{s}" / "ses-01" / "anat" / f"sub-{s}_T1w.json")) for s in qpn["PATNO"]}
qc = pd.DataFrame({m: {s: d[m] for s, d in iqm.items()} for m in fsvv.MRIQC_METRICS})
qc_corr = fsvv.qc_correlations(qpn_table.join(qc), pairs=QPN_PAIRS)
qc_corr.to_csv(fsvv.out("qpn_mriqc_variability_correlations.csv"), index=False)
qc_corr["significant"] = qc_corr["p"] < fsvv.ALPHA / (qc_corr["kind"].map(fsvv.N_TESTS) * len(fsvv.MRIQC_METRICS))
qc_corr.groupby(["metric", "kind"])["significant"].sum().unstack("kind")